<a href="https://colab.research.google.com/github/chetools/CHE4071_Spring2026/blob/main/TuningMethods.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!wget -N -q https://raw.githubusercontent.com/chetools/chetools/main/tools/che5.ipynb -O che5.ipynb
%run che5.ipynb

In [ ]:
#Ziegler Nichols closed-loop.

Kcu =17.85
Tu = 0.35
Kc = 0.45*Kcu
taui = 0.83*Tu

K = 1.
theta = 0.1
tau = 1.
Gc = Kc*(1 + 1/(taui*s))

Gp = K*exp(-theta*s)/(tau*s + 1)

Gclosed = Gc*Gp/(1+Gc*Gp)
t, h = sim(Gclosed, lambda t: 1, N=1000, dt=0.01)
fig=make_subplots()
fig.add_scatter(x=t, y=h, mode='lines')

In [ ]:
#Smith-Corripio ITAE, open-loop tuning method
pa,pb = 0.586, -0.916
ia,ib = 1.03, -0.165
Y = pa*(theta/tau)**pb
Kc = Y/K
taui = tau/(ia + ib*(theta/tau))

In [ ]:
Gc = Kc*(1 + 1/(taui*s))
Gp = K*exp(-theta*s)/(tau*s + 1)
Gclosed = Gc*Gp/(1+Gc*Gp)
t, h = sim(Gclosed, lambda t: 1, N=1000, dt=0.01)
fig=make_subplots()
fig.add_scatter(x=t, y=h, mode='lines')

In [ ]:
#Approximating a 4th system with a FOPTD model
Gp = 1.666/((3*s+1)*(1.5*s+1)*(4.6*s+1)*(5*s+1))
t, hexp = sim(Gp, lambda t: 1, N=50, dt=1.)

In [ ]:
def sse(v):
    K, tau, theta = v
    model = np.where(t<theta, 0., 1.65*(1-np.exp(-(t-theta)/tau)))
    return np.sum(np.abs(hexp - model))

In [ ]:
Kfit, taufit, thetafit = sp.optimize.minimize(sse, (1.6, 10., 4.)).x

In [ ]:

tau = 10.
theta = 4.
model = np.where(t<thetafit, 0., Kfit*(1-np.exp(-(t-thetafit)/taufit)))
fig=make_subplots()
fig.add_scatter(x=t, y=hexp, mode='markers')
fig.add_scatter(x=t, y=model, mode='lines')